In [3]:
import pandas as pd

# Load the cleaned CSV file
df = pd.read_csv("ncr_ride_bookings_with_weather_filled_scaled_short.csv")

# Display the first 5 rows to preview the data structure
df.head()


,Date,Time,booking_datetime,booking_hour,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,...,wind_speed_10m_dropoff_scaled.1,precipitation_log_scaled,precipitation_log_scaled.1,rain_log_scaled,rain_log_scaled.1,snowfall_log_scaled,snowfall_log_scaled.1,precipitation_dropoff_log_scaled,rain_dropoff_log_scaled,snowfall_dropoff_log_scaled
0,2024-03-23,12:29:38,2024-03-23 12:29,2024-03-23 12:00,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,...,1.392857,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1,2024-11-29,18:01:39,2024-11-29 18:01,2024-11-29 18:00,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,...,-1.250000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
2,2024-08-23,8:56:10,2024-08-23 8:56,2024-08-23 8:00,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,...,-0.464286,0.182322,0.182322,0.182322,0.182322,0.0,0.0,0.0,0.0,0.0
3,2024-10-21,17:17:25,2024-10-21 17:17,2024-10-21 17:00,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,...,-0.214286,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
4,2024-09-16,22:08:00,2024-09-16 22:08,2024-09-16 22:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,...,0.017857,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


# 🚖 Uber Ride Completion Prediction (Supervised Learning)

This notebook builds a supervised learning pipeline to predict **Booking Status**  
(Completed vs Cancelled/Incomplete) from Uber ride + weather data.  

follow a step-by-step pipeline:

1. Load & clean the dataset  
2. Select valid features  
3. Build preprocessing steps  
4. Define models to try  
5. Map labels & setup cross-validation  
6. Train & evaluate models  
7. Export results and summarize  

In [4]:
# =============================
# Cell 1) Imports, load, folders, minimal cleaning (simplified)
# =============================
from pathlib import Path
import pandas as pd

# Try multiple possible dataset paths
csv_path = next(
    (p for p in [
        Path("datasets/processed/ncr_ride_bookings_with_weather_filled_scaled_short.csv"),
        Path("ncr_ride_bookings_with_weather_filled_scaled_short.csv"),
    ] if p.exists()),
    None,
)
if csv_path is None:
    raise FileNotFoundError("Cannot find processed CSV at expected paths.")

# Ensure artifact directories exist
for d in ("artifacts/metrics", "artifacts/plots", "artifacts/models"):
    Path(d).mkdir(parents=True, exist_ok=True)

# Load dataframe
df = pd.read_csv(csv_path, low_memory=False)

# Deduplicate duplicate column names (keep first)
before = df.shape[1]
df = df.loc[:, ~df.columns.duplicated(keep="first")]
after = df.shape[1]
if after != before:
    print(f"Deduplicated columns: {before} -> {after}")

# Convert *_missing_flag columns to 0/1
flag_cols = [c for c in df.columns if c.endswith("_missing_flag")]
if flag_cols:
    df[flag_cols] = (pd.to_numeric(df[flag_cols].stack(), errors="coerce")
                       .fillna(0).astype("int8").unstack())

# Force known numeric-like columns to numeric (avoid 'object')
numeric_like_raw = {
    "booking_hour",
    "pick_longitude", "pick_latitude",
    "drop_longitude", "drop_latitude",
    "pick_station_latitude", "pick_station_longitude",
    "drop_station_latitude", "drop_station_longitude",
}
num_like_cols = [c for c in df.columns
                 if c.endswith(("_scaled", "_log_scaled")) or c in numeric_like_raw]
if num_like_cols:
    df[num_like_cols] = df[num_like_cols].apply(pd.to_numeric, errors="coerce")

print("Folders & dataframe ready.")
df.head()


Folders & dataframe ready.


,Date,Time,booking_datetime,booking_hour,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,...,wind_speed_10m_dropoff_scaled.1,precipitation_log_scaled,precipitation_log_scaled.1,rain_log_scaled,rain_log_scaled.1,snowfall_log_scaled,snowfall_log_scaled.1,precipitation_dropoff_log_scaled,rain_dropoff_log_scaled,snowfall_dropoff_log_scaled
0,2024-03-23,12:29:38,2024-03-23 12:29,NaN,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,...,1.392857,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1,2024-11-29,18:01:39,2024-11-29 18:01,NaN,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,...,-1.250000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
2,2024-08-23,8:56:10,2024-08-23 8:56,NaN,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,...,-0.464286,0.182322,0.182322,0.182322,0.182322,0.0,0.0,0.0,0.0,0.0
3,2024-10-21,17:17:25,2024-10-21 17:17,NaN,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,...,-0.214286,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
4,2024-09-16,22:08:00,2024-09-16 22:08,NaN,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,...,0.017857,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


## 📘 Cell 1 — Load Data & Prepare
- Try multiple paths to locate the processed dataset  
- Ensure artifact folders (`metrics`, `plots`, `models`) exist  
- Load the dataset into a DataFrame  
- Deduplicate duplicate column names  
- Convert `*_missing_flag` columns to binary (0/1)  
- Force numeric-like columns (e.g. scaled values, coordinates, booking_hour) to numeric  

👉 **Goal:** Start with a clean dataset that won’t break models later.  


In [5]:
# =============================
# Cell 2) Feature inference
# =============================
from typing import List, Tuple
import pandas as pd

TARGET_COL = "Booking Status"

NUMERIC_RAW_WHITELIST = [
    "booking_hour",
    "pick_longitude", "pick_latitude",
    "drop_longitude", "drop_latitude",
    "pick_station_latitude", "pick_station_longitude",
    "drop_station_latitude", "drop_station_longitude",
]

EXCLUDE_ALWAYS = {
    "Booking ID", "Customer ID",
    "Date", "Time", "booking_datetime",
    "Pickup Location", "Drop Location",
    "pick_address", "drop_address",
    "pick_region", "drop_region",
    "pick_locality", "drop_locality",
    TARGET_COL,
}

_SUFFIX_NUMERIC = ("_scaled", "_log_scaled")
_FLAG_SUFFIX = "_missing_flag"

def _uniq(seq: List[str]) -> List[str]:
    return list(dict.fromkeys(seq))  # preserve order, remove duplicates

def infer_feature_lists(df: pd.DataFrame, target_col: str) -> Tuple[List[str], List[str], List[str]]:
    cols = [c for c in df.columns if c not in EXCLUDE_ALWAYS]
    is_num = df.dtypes.apply(lambda dt: pd.api.types.is_numeric_dtype(dt))

    scaled = [c for c in cols if c.endswith(_SUFFIX_NUMERIC)]
    fill   = [c for c in cols if c.endswith("_fill")]
    flags  = [c for c in cols if c.endswith(_FLAG_SUFFIX)]
    cat_seed = ["Vehicle Type"] if "Vehicle Type" in df.columns else []

    numeric_fill     = [c for c in fill if is_num.get(c, False)]
    categorical_fill = [c for c in fill if not is_num.get(c, False)]
    numeric_raw = [c for c in NUMERIC_RAW_WHITELIST if c in df.columns]

    numeric_features     = _uniq(scaled + numeric_fill + numeric_raw)
    categorical_features = _uniq(cat_seed + categorical_fill)
    missing_flags        = _uniq(flags)

    bad_numeric = [c for c in numeric_features if not is_num.get(c, False)]
    if bad_numeric:
        print("⚠️ Dropped non-numeric from numeric_features:", bad_numeric)
        numeric_features = [c for c in numeric_features if c not in bad_numeric]

    print(f"#numeric={len(numeric_features)}  #categorical={len(categorical_features)}  #flags={len(missing_flags)}")
    print("Numeric sample:", numeric_features[:10])
    print("Categorical sample:", categorical_features[:10])
    print("Flags sample:", missing_flags[:10])

    return numeric_features, categorical_features, missing_flags

numeric_features, categorical_features, missing_flags = infer_feature_lists(df, TARGET_COL)

total_feats = len(numeric_features) + len(categorical_features) + len(missing_flags)
print("Numeric FULL:", numeric_features)
print("Categorical FULL:", categorical_features)
print("Flags FULL:", missing_flags)
print("TOTAL features:", total_feats)


#numeric=34  #categorical=4  #flags=3
Numeric sample: ['Avg VTAT_fill_scaled', 'Avg CTAT_fill_scaled', 'Booking Value_fill_scaled', 'Ride Distance_fill_scaled', 'temperature_2m_scaled', 'relative_humidity_2m_scaled', 'dew_point_2m_scaled', 'apparent_temperature_scaled', 'wind_speed_10m_scaled', 'temperature_2m_dropoff_scaled']
Categorical sample: ['Vehicle Type', 'Reason for cancelling by Customer_fill', 'Driver Cancellation Reason_fill', 'Incomplete Rides Reason_fill']
Flags sample: ['VTAT_missing_flag', 'CTAT_missing_flag', 'BookingValue_missing_flag']
Numeric FULL: ['Avg VTAT_fill_scaled', 'Avg CTAT_fill_scaled', 'Booking Value_fill_scaled', 'Ride Distance_fill_scaled', 'temperature_2m_scaled', 'relative_humidity_2m_scaled', 'dew_point_2m_scaled', 'apparent_temperature_scaled', 'wind_speed_10m_scaled', 'temperature_2m_dropoff_scaled', 'relative_humidity_2m_dropoff_scaled', 'dew_point_2m_dropoff_scaled', 'apparent_temperature_dropoff_scaled', 'wind_speed_10m_dropoff_scaled', 'precipi

## 📘 Cell 2 — Feature Inference
- Define which columns are always excluded (IDs, text, target, raw datetime fields)  
- Whitelist known numeric raw features (booking_hour, coordinates)  
- Automatically infer features:
  - **Numeric:** scaled, log_scaled, numeric *_fill, whitelisted raw  
  - **Categorical:** non-numeric *_fill, Vehicle Type  
  - **Flags:** *_missing_flag  
- Validate numeric columns are truly numeric  

👉 **Goal:** Build clean feature lists for modeling.  


In [6]:
# =============================
# Cell 3) Build preprocessor
# =============================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

def build_preprocessor(
    numeric_cols: List[str],
    categorical_cols: List[str],
    flag_cols: List[str]
) -> ColumnTransformer:
    transformers = []

    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=False, min_frequency=0.01)),
            ]),
            categorical_cols,
        ))

    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
            ]),
            numeric_cols,
        ))

    if flag_cols:
        transformers.append(("flag", "passthrough", flag_cols))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

preprocessor = build_preprocessor(numeric_features, categorical_features, missing_flags)


## 📘 Cell 3 — Build Preprocessor
- Create a ColumnTransformer with three parts:
  1. **Categorical:** impute missing with most frequent, then one-hot encode  
  2. **Numeric:** impute missing with median  
  3. **Flags:** passthrough as-is (already 0/1)  

👉 **Goal:** Ensure the model receives a clean numeric feature matrix for training.  


In [7]:
# =============================
# Cell 4) Model registry
# =============================
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def get_models(seed: int = 42):
    """Baseline + Logistic Regression + Decision Tree + Ensembles"""
    return {
        "dummy_mf": DummyClassifier(strategy="most_frequent"),
        "logreg_l2": LogisticRegression(
            solver="saga",
            penalty="l2",
            max_iter=1000,
            class_weight="balanced",
            random_state=seed
        ),
        "dtree": DecisionTreeClassifier(
            min_samples_split=5,
            class_weight="balanced",
            random_state=seed
        ),
        "rf_300": RandomForestClassifier(
            n_estimators=300,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=seed
        ),
        "gbdt": GradientBoostingClassifier(random_state=seed),
    }

MODELS = get_models()


## 📘 Cell 4 — Model Registry
- Define a dictionary of models to evaluate:
  - Dummy classifier (baseline)  
  - Logistic Regression (balanced, saga solver)  
  - Decision Tree (balanced)  
  - Random Forest (300 trees, balanced_subsample)  
  - Gradient Boosted Trees  
- Enables consistent looping through models.  

👉 **Goal:** Compare performance across multiple model families.  


In [8]:
# =============================
# Cell 5) Exact-label mapping & safer CV
# =============================
from typing import Iterable, Set
from sklearn.model_selection import StratifiedKFold
import numpy as np

POSITIVE_LABELS: Set[str] = frozenset({
    "Cancelled by Customer",
    "Cancelled by Driver",
    "Incomplete",
    "No Driver Found",
})
NEGATIVE_LABELS: Set[str] = frozenset({"Completed"})

def map_status_to_binary_exact_labels(s: object, positive: Set[str] = POSITIVE_LABELS) -> int:
    """Return 1 if s ∈ positive set, else 0 (NaN -> 0)."""
    return int(pd.notna(s) and str(s).strip() in positive)

def validate_labels(y_text: pd.Series,
                    pos: Set[str] = POSITIVE_LABELS,
                    neg: Set[str] = NEGATIVE_LABELS) -> None:
    """Print unknown labels (not in pos ∪ neg)."""
    unknown = sorted(set(y_text.astype(str).str.strip()) - (pos | neg))
    if unknown:
        print("⚠️ Unknown Booking Status values:", unknown)

def safe_stratified_cv(y_bin: pd.Series, n_splits: int = 5, seed: int = 42):
    """Stratified K-Fold with safeguards for small minority classes."""
    vc = y_bin.value_counts()
    if len(vc) < 2:
        raise ValueError("Only one class present overall after mapping.")

    n = max(min(int(vc.min()), n_splits), 2)
    skf = StratifiedKFold(n_splits=n, shuffle=True, random_state=seed)

    for i, (tr, va) in enumerate(skf.split(np.zeros(len(y_bin)), y_bin), 1):
        y_tr, y_va = y_bin.iloc[tr], y_bin.iloc[va]
        if y_tr.nunique() < 2 or y_va.nunique() < 2:
            print(f"⚠️ Skipping fold {i}: single class in train/valid.")
            continue
        yield i, tr, va


## 📘 Cell 5 — Label Mapping & Safe CV
- Define positive labels (all cancellations/incompletes) vs negative labels (Completed)  
- Provide a helper to map Booking Status → binary labels  
- Validate labels against known sets (warn if unexpected values exist)  
- Safe StratifiedKFold:
  - Reduces folds if minority class is too small  
  - Skips folds with single-class splits  

👉 **Goal:** Prepare labels and cross-validation in a safe, robust way.  


In [9]:
# =============================
# Cell 6) Evaluate with CV and collect metrics
# =============================
import uuid
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
import pandas as pd

X = df.drop(columns=[TARGET_COL])
y_text = df[TARGET_COL].astype(str).str.strip()

# 1) Validate and map labels
validate_labels(y_text)  # uses default POSITIVE/NEGATIVE sets
y = y_text.map(map_status_to_binary_exact_labels).astype(int)
print("Label distribution:", y.value_counts().to_dict())

# 2) Track run info and preprocessor summary
run_id = str(uuid.uuid4())
ts = pd.Timestamp.utcnow().isoformat()
print(f"Preprocessor columns: num={len(numeric_features)}, cat={len(categorical_features)}, flags={len(missing_flags)}")

rows = []
for model_name, clf in MODELS.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])

    for fold_id, tr_idx, va_idx in safe_stratified_cv(y, n_splits=5, seed=42):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_va)

        # Base metrics
        metrics = {
            "accuracy":  accuracy_score(y_va, y_pred),
            "precision": precision_score(y_va, y_pred, zero_division=0),
            "recall":    recall_score(y_va, y_pred, zero_division=0),
            "f1":        f1_score(y_va, y_pred, zero_division=0),
        }

        # ROC-AUC only if model supports probabilities and both classes are present
        if hasattr(pipe.named_steps["clf"], "predict_proba") and y_va.nunique() == 2:
            prob1 = pipe.predict_proba(X_va)[:, 1]
            metrics["roc_auc"] = roc_auc_score(y_va, prob1)

        params = clf.get_params() if hasattr(clf, "get_params") else {}
        rows.extend(
            {
                "run_id": run_id,
                "timestamp": ts,
                "model_name": model_name,
                "fold": fold_id,
                "metric": m,
                "value": float(v),
                "params": str(params),
            }
            for m, v in metrics.items()
        )

metrics_df = pd.DataFrame(rows)
print(f"Collected {len(metrics_df)} metric rows.")
metrics_df.head()


Label distribution: {0: 90896, 1: 55718}
Preprocessor columns: num=34, cat=4, flags=3


c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. 

Collected 125 metric rows.


,run_id,timestamp,model_name,fold,metric,value,params
0,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,accuracy,0.619957,"{'constant': None, 'random_state': None, 'stra..."
1,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,precision,0.000000,"{'constant': None, 'random_state': None, 'stra..."
2,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,recall,0.000000,"{'constant': None, 'random_state': None, 'stra..."
3,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,f1,0.000000,"{'constant': None, 'random_state': None, 'stra..."
4,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,roc_auc,0.500000,"{'constant': None, 'random_state': None, 'stra..."


## 📘 Cell 6 — Train & Evaluate
- Run cross-validation for each model  
- For each fold:
  - Train the model  
  - Predict on validation data  
  - Collect metrics:
    - Accuracy (overall correctness)  
    - Precision (how many predicted cancels were real cancels)  
    - Recall (how many real cancels we caught)  
    - F1 (balance of precision & recall)  
    - ROC-AUC (how well model separates classes, if probabilities available)  
- Save results in a long-format DataFrame (model × fold × metric).  

👉 **Goal:** Evaluate model performance fairly and consistently.  


In [10]:
# =============================
# Cell 7) Export metrics + feature audit
# =============================
out_path = "artifacts/metrics/metrics.csv"
metrics_df.to_csv(out_path, index=False)
print(f"Saved metrics: {out_path}  shape={metrics_df.shape}")

# audit feature space size (after preprocessing)
from sklearn.pipeline import Pipeline
try:
    prep_only = Pipeline([("prep", preprocessor)])
    prep_only.fit(X, y)
    try:
        feat_names = prep_only.named_steps["prep"].get_feature_names_out()
        print(f"Final transformed feature count: {len(feat_names)}")
        # pd.Series(feat_names, name="feature").to_csv("artifacts/metrics/features_used.csv", index=False)
    except Exception as e:
        print("Could not fetch feature names out:", e)
except Exception as e:
    print("Feature audit skipped:", e)

# Quick F1 leaderboard
leaderboard = (
    metrics_df.loc[metrics_df["metric"] == "f1", ["model_name", "value"]]
    .groupby("model_name")["value"]
    .agg(mean="mean", std="std", count="count")
    .sort_values("mean", ascending=False)
    .reset_index()
)
leaderboard


Saved metrics: artifacts/metrics/metrics.csv  shape=(125, 7)


c:\Users\yauli\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Final transformed feature count: 58


,model_name,mean,std,count
0,dtree,1.0,0.0,5
1,gbdt,1.0,0.0,5
2,logreg_l2,1.0,0.0,5
3,rf_300,1.0,0.0,5
4,dummy_mf,0.0,0.0,5


## 📘 Cell 7 — Export & Summarize
- Save metrics DataFrame to `artifacts/metrics/metrics.csv`  
- Quick leaderboard: summarize F1 scores (mean ± std per model).  

👉 **Goal:** Provide teammates (Visualization) with a ready-to-use CSV and a quick leaderboard.  


In [11]:
# -----------------------------
#  (Cell 8) Peek metrics
# -----------------------------
pd.read_csv("artifacts/metrics/metrics.csv")

,run_id,timestamp,model_name,fold,metric,value,params
0,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,accuracy,0.619957,"{'constant': None, 'random_state': None, 'stra..."
1,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,precision,0.000000,"{'constant': None, 'random_state': None, 'stra..."
2,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,recall,0.000000,"{'constant': None, 'random_state': None, 'stra..."
3,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,f1,0.000000,"{'constant': None, 'random_state': None, 'stra..."
4,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,dummy_mf,1,roc_auc,0.500000,"{'constant': None, 'random_state': None, 'stra..."
...,...,...,...,...,...,...,...
120,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,gbdt,5,accuracy,1.000000,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'..."
121,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,gbdt,5,precision,1.000000,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'..."
122,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,gbdt,5,recall,1.000000,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'..."
123,5460a59e-0ada-4759-a203-41e17539f9fa,2025-09-21T00:10:08.508741+00:00,gbdt,5,f1,1.000000,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'..."
